<a href="https://colab.research.google.com/github/papertuc2000/CL-Drive/blob/dev/Remove_Nan_All.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.signal import butter, iirnotch, filtfilt
import warnings
import time # برای اندازه‌گیری زمان

warnings.filterwarnings("ignore", category=FutureWarning, message=".*fillna with 'method' is deprecated.*")

# ==========================================
# CONFIGURATION
# ==========================================

dataset_path = '/content/drive/MyDrive/Colab Notebooks/CL-Drive'
modalities = ['EEG', 'ECG', 'EDA', 'Gaze']
clean_data_lists = {m: [] for m in modalities}

SAMPLING_RATES = {
    'EEG': 256,
    'ECG': 512,
    'EDA': 128,
    'Gaze': 50
}

FILTER_PARAMS = {
    'EEG': {'low': 0.4, 'high': 75.0, 'notch': 60.0, 'q': 30.0},
    'ECG': {'low': 5.0, 'high': 15.0, 'notch': None},
    'EDA': {'low': 0.05, 'high': 3.0, 'notch': None},
    'Gaze': None
}

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def butter_bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    if low >= 1.0 or high >= 1.0 or low <= 0:
        return data
    try:
        b, a = butter(order, [low, high], btype='band')
        if len(data.shape) == 1:
            return filtfilt(b, a, data)
        else:
            return np.apply_along_axis(lambda x: filtfilt(b, a, x), axis=0, arr=data)
    except Exception:
        return data

def apply_notch_filter(data, freq, fs, q=30):
    try:
        b, a = iirnotch(freq, Q=q, fs=fs)
        if len(data.shape) == 1:
            return filtfilt(b, a, data)
        else:
            return np.apply_along_axis(lambda x: filtfilt(b, a, x), axis=0, arr=data)
    except Exception:
        return data

# ==========================================
# MAIN PROCESSING LOOP
# ==========================================

print("="*70)
print("STARTING OPTIMIZED MULTIMODAL PREPROCESSING")
print("="*70)

for mod in modalities:
    print(f"\n--- Processing Modality: {mod} ---")

    pattern = os.path.join(dataset_path, mod, '*', f'{mod.lower()}_*_level_*.csv')
    files = glob(pattern)

    if not files:
        pattern = os.path.join(dataset_path, mod, '*', '*.csv')
        files = glob(pattern)

    if not files:
        print(f"Warning: No files found for {mod}. Skipping.")
        continue

    success_count = 0
    empty_count = 0
    skipped_baseline_count = 0

    # مرتب‌سازی فایل‌ها برای پردازش منظم
    files.sort()

    for file_path in files:
        filename = os.path.basename(file_path)

        # نمایش نام فایل قبل از پردازش (برای دیباگ کردن توقف‌ها)
        print(f"Processing: {filename}...", end=" ")

        # 1. SKIP BASELINE FILES
        if "baseline" in filename.lower():
            print("Skipped (Baseline)")
            skipped_baseline_count += 1
            continue

        start_time = time.time()

        try:
            # OPTIMIZED READING:
            # 1. dtype=float: فرض می‌کنیم همه داده‌ها عددی هستند (سرعت بسیار بالا)
            # 2. engine='c': استفاده از موتور سریع C
            # 3. encoding: تلاش برای utf-8، اگر نشد latin-1
            try:
                df_raw = pd.read_csv(
                    file_path,
                    header=0,
                    dtype=float,
                    engine='c',
                    encoding='utf-8',
                    on_bad_lines='skip' # رد کردن خطوط خراب به جای خطا دادن
                )
            except UnicodeDecodeError:
                # اگر utf-8 جواب نداد، با latin-1 امتحان کن
                df_raw = pd.read_csv(
                    file_path,
                    header=0,
                    dtype=float,
                    engine='c',
                    encoding='latin-1',
                    on_bad_lines='skip'
                )

            df_numeric = df_raw # چون همه را float خواندیم، دیگر نیازی به select_dtypes نیست

            if df_numeric.empty:
                print("Skipped (Empty)")
                continue

            original_rows = len(df_numeric)

            # Separate Timestamp and Signal Data
            if df_numeric.shape[1] > 1:
                timestamps = df_numeric.iloc[:, 0].values
                signal_data = df_numeric.iloc[:, 1:].values
                signal_cols = df_numeric.columns[1:]
            else:
                timestamps = None
                signal_data = df_numeric.values
                signal_cols = df_numeric.columns

            processed_signals = signal_data.copy()

            # Apply Filters
            if mod in FILTER_PARAMS and FILTER_PARAMS[mod]:
                params = FILTER_PARAMS[mod]
                fs = SAMPLING_RATES[mod]
                processed_signals = butter_bandpass_filter(processed_signals, params['low'], params['high'], fs, order=2)
                if params.get('notch'):
                    processed_signals = apply_notch_filter(processed_signals, params['notch'], fs, params.get('q', 30))

            # Reconstruct DataFrame
            if timestamps is not None:
                df_processed = pd.DataFrame(processed_signals, columns=signal_cols)
                df_processed.insert(0, 'Timestamp', timestamps)
            else:
                df_processed = pd.DataFrame(processed_signals, columns=signal_cols)

            # HANDLE MISSING VALUES
            df_clean = pd.DataFrame()

            if mod == 'EEG':
                df_clean = df_processed.dropna(how='any')

            elif mod == 'ECG':
                df_clean = df_processed.interpolate(method='polynomial', order=5, limit_area='inside')
                df_clean = df_clean.dropna()

            elif mod in ['EDA', 'Gaze']:
                df_clean = df_processed.ffill()
                df_clean = df_clean.bfill()
                df_clean = df_clean.dropna()

            final_array = df_clean.values
            cleaned_rows = len(df_clean)

            if cleaned_rows == 0:
                print(f"Warning: Became EMPTY after cleaning!")
                empty_count += 1
                continue

            clean_data_lists[mod].append(final_array)
            success_count += 1

            elapsed = time.time() - start_time
            print(f"OK ({original_rows}->{cleaned_rows}) in {elapsed:.2f}s")

            # اختیاری: نمایش چند سطر اول فقط برای اولین فایل موفق هر مدالیته
            if success_count == 1:
                print(df_clean.head().to_string(index=False))

        except Exception as e:
            elapsed = time.time() - start_time
            print(f"ERROR after {elapsed:.2f}s: {str(e)[:50]}...")

    print(f"[{mod}] Summary: {success_count} successful, {skipped_baseline_count} baselines skipped, {empty_count} empty.")

# ==========================================
# FINAL SUMMARY
# ==========================================
print("\n" + "="*70)
print("PREPROCESSING COMPLETE")
print("="*70)

total_files_processed = 0
for mod in modalities:
    count = len(clean_data_lists[mod])
    total_files_processed += count
    if count > 0:
        total_rows = sum(arr.shape[0] for arr in clean_data_lists[mod])
        cols = clean_data_lists[mod][0].shape[1]
        print(f"[{mod}] Success: {count} files | Total Valid Rows: {total_rows:,} | Cols: {cols}")
    else:
        print(f"[{mod}] FAILED: No data loaded.")

if total_files_processed > 0:
    print(f"\nTotal files successfully processed: {total_files_processed}")
    print("Data is ready for segmentation.")
else:
    print("\nCRITICAL: No data was processed.")

STARTING OPTIMIZED MULTIMODAL PREPROCESSING

--- Processing Modality: EEG ---
Processing: eeg_baseline_level_1.csv... Skipped (Baseline)
Processing: eeg_baseline_level_2.csv... Skipped (Baseline)
Processing: eeg_baseline_level_3.csv... Skipped (Baseline)
Processing: eeg_baseline_level_4.csv... Skipped (Baseline)
Processing: eeg_baseline_level_5.csv... Skipped (Baseline)
Processing: eeg_baseline_level_6.csv... Skipped (Baseline)
Processing: eeg_baseline_level_7.csv... Skipped (Baseline)
Processing: eeg_baseline_level_8.csv... Skipped (Baseline)
Processing: eeg_baseline_level_9.csv... Skipped (Baseline)
Processing: eeg_data_level_1.csv... OK (46594->46594) in 0.12s
 Timestamp       TP9        AF7       AF8      TP10
120.007812 -1.990833 -10.128303  8.769133 -0.892351
120.011719 14.320608  -4.517761 15.337568 -0.107444
120.015625 20.500738 -10.328422 31.705631 -1.336816
120.019531 12.993641 -19.734923 40.361976  7.110694
120.023438 15.674398 -17.639489 32.959161 19.746439
Processing: eeg_